# Теомор — бесшовная текстура в Colab

**Что делать:** жми ▶ на каждой ячейке **сверху вниз**. Не пропускай.

Перед стартом: меню **Среда выполнения → Сменить среду выполнения → GPU**.


## 1) Проверка GPU


In [ ]:
import torch
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
else:
    raise SystemExit("GPU выключен. Среда выполнения → Сменить → GPU → снова ▶")


## 2) Установка библиотек (3–7 мин, один раз за сессию)


In [ ]:
# Colab сейчас часто на Python 3.13 — пиним версии, иначе PIL._Ink ломает diffusers
%pip -q uninstall -y pillow pillow-simd 2>/dev/null
%pip -q install -U "pillow==10.4.0" "diffusers==0.30.3" "transformers==4.44.2" "accelerate==0.34.2" "safetensors==0.4.5" "huggingface_hub==0.25.2"
print("OK install. Если снова ImportError PIL — Среда выполнения → Перезапустить сеанс, GPU, потом с ячейки 1.")


## 3) Загрузка модели

Модель скачается один раз за сессию (~2–4 ГБ). Жди без паники.


In [ ]:
import torch
from diffusers import StableDiffusionPipeline

# на всякий случай после рестарта
assert torch.cuda.is_available(), "Включи GPU"

pipe = StableDiffusionPipeline.from_pretrained(
    "sd-legacy/stable-diffusion-v1-5",  # зеркало runwayml, меньше сюрпризов с доступом
    torch_dtype=torch.float16,
    safety_checker=None,
    requires_safety_checker=False,
)
pipe = pipe.to("cuda")
pipe.enable_attention_slicing()
print("model ready")


## 4) Промпт Теомор (можно менять SLOT)

Слоты и якорь также в репо: `godot/docs/AI_ASSETS.md`.


In [ ]:
ANCHOR = (
    "Teomor style: dirty parchment yellow, diseased purple, black ink strokes, "
    "one dominant color, PS2 Silent Hill + Dread Delusion grit, hand-painted look, "
    "halftone grain, desaturated, never clean PBR, never neon, never glossy, "
    "seamless tileable texture, flat albedo, no perspective, no characters, no UI"
)

NEGATIVE = (
    "clean PBR, photoreal, sharp 8k, neon, anime, plastic, glossy, chrome, bloom, "
    "depth of field, text, logo, border, character, face, perspective room"
)

# --- выбери ОДИН SLOT ---
SLOT = (
    "seamless tileable damp alley wall albedo, diseased muddy purple dominant, "
    "wet dirt, mushroom spore stains, black ink puddles, sickly paper in cracks, "
    "Silent Hill fog grit, hand painted, not photo cobblestone"
)
# другие SLOT: attic plaster / floorboards / mushroom cap — см. AI_ASSETS.md

prompt = ANCHOR + ". " + SLOT
print(prompt[:200], "...")


## 5) Генерация 512×512


In [ ]:
import random
seed = random.randint(1, 999999)
generator = torch.Generator("cuda").manual_seed(seed)
print("seed", seed)

result = pipe(
    prompt=prompt,
    negative_prompt=NEGATIVE,
    width=512,
    height=512,
    num_inference_steps=28,
    guidance_scale=6.5,
    generator=generator,
)
img = result.images[0]
img.save("/content/teomor_tile_raw.png")
display(img)
print("saved /content/teomor_tile_raw.png")


## 6) Сделать бесшовнее (сведение краёв)

Не заменяет идеальный hand-paint, но убирает грубый шов.


In [ ]:
import numpy as np
from PIL import Image

def make_seamless(im: Image.Image, blend: int = 48) -> Image.Image:
    a = np.asarray(im.convert("RGB"), dtype=np.float32) / 255.0
    h, w, _ = a.shape
    b = min(blend, h // 4, w // 4)
    out = a.copy()
    for i in range(b):
        t = i / max(1, b - 1)
        wgt = 0.5 - 0.5 * np.cos(np.pi * t)
        out[:, i] = a[:, i] * (1 - wgt) + a[:, w - 1 - i] * wgt
        out[:, w - 1 - i] = a[:, w - 1 - i] * (1 - wgt) + a[:, i] * wgt
        out[i] = out[i] * (1 - wgt) + out[h - 1 - i] * wgt
        out[h - 1 - i] = out[h - 1 - i] * (1 - wgt) + out[i] * wgt
    return Image.fromarray((np.clip(out, 0, 1) * 255).astype(np.uint8))

raw = Image.open("/content/teomor_tile_raw.png")
seamless = make_seamless(raw, blend=56)
seamless.save("/content/teomor_tile_seamless.png")
display(seamless)
print("saved /content/teomor_tile_seamless.png")
print("Скачай файл: слева папка Files → teomor_tile_seamless.png → ⋮ → Download")


## 7) Ещё вариант

Снова ▶ ячейку **5**, потом **6** — будет другой seed.

Когда понравится — скачай PNG и кинь в чат / в `Текстурки/` на git.
